# Đội 34 – Nguyễn Bá Duy (`TEAM_ID = 39`): Pipeline Tạo Dữ Liệu Kaggle (All-in-One)

**Chỉ cần tải duy nhất tệp `.ipynb` này lên Kaggle (`File → Import Notebook`)!**
Toàn bộ mã nguồn `src/`, `configs/` và `train_and_export.ipynb` đã được nhúng sẵn bên trong notebook.

### Thiết lập bắt buộc trên Kaggle (Cột bên phải → Notebook Options)
1. **Accelerator**: `GPU T4 x2` (hoặc `GPU P100`)
2. **Internet**: `ON` (bắt buộc để cài gói và tải HuggingFace / `sam2_b.pt` / `yolo11x-seg.pt`)
3. *(Khuyên dùng — 0 tốn dung lượng `/kaggle/working`)* Bấm **+ Add Input** thêm sẵn các Kaggle Datasets nếu muốn dùng:
   - **IDD**: `abhisheksukhwal/indian-driving-dataset-segmentation-part-2` hoặc `theainerd/indian-driving-dataset`
   - **BDD100K**: `solarmainframe/bdd100k` hoặc `marquis03/bdd100k-images`
   - **Ảnh/Video dashcam VN tự thu thập** (nếu có)

Sau khi chạy xong notebook này, bạn sẽ có **`/kaggle/working/du_lieu_v1.zip`** và **`/kaggle/working/master_v1/`** để nạp thẳng vào **`train_and_export.ipynb`** (`TEAM_ID = 39`).


In [ ]:
#@title 1. Tham số Đội 34 & Tùy chọn nguồn dữ liệu { display-mode: "form" }
TEAM_NAME = "Đội 34 – Nguyễn Bá Duy"
TEAM_ID = 39  # Seed cố định của đội

# ---- Thư mục làm việc (giữ /kaggle/tmp cho dữ liệu thô để không tràn giới hạn 20GB của /kaggle/working) ----
import os
from pathlib import Path

IS_KAGGLE = Path("/kaggle/working").exists()
REPO_DIR = Path("/kaggle/working/repo") if IS_KAGGLE else Path("work_kaggle/repo")
TMP_RAW_DIR = Path("/kaggle/tmp/raw") if IS_KAGGLE else Path("work_kaggle/tmp_raw")
STAGED_DIR = Path("/kaggle/working/staged") if IS_KAGGLE else Path("work_kaggle/staged")
GOLDEN_VAL_DIR = Path("/kaggle/working/golden_val") if IS_KAGGLE else Path("work_kaggle/golden_val")
MASTER_DIR = Path("/kaggle/working/master_v1") if IS_KAGGLE else Path("work_kaggle/master_v1")
OUT_ZIP_PATH = Path("/kaggle/working/du_lieu_v1.zip") if IS_KAGGLE else Path("work_kaggle/du_lieu_v1.zip")

# ---- Bật/tắt các nguồn Open-Source ----
INSTALL_DEPS = True

# Nguồn 1: Biển báo Việt Nam trên HuggingFace (BBox -> SAM2 mask, tự cắt tràn cột)
RUN_VNTS_HF = True
VNTS_HF_REPO = "star092304/Traffic-sign-detection-VietNam"
VNTS_MAX_IMAGES = 1500   # Tăng lên 3000-5000 ở các vòng sau; vòng 1 giữ 1500 để chạy xong trong ~5 phút
SAM2_MODEL = "sam2_b.pt"

# Nguồn 2: Dashcam VN trên HuggingFace hoặc tự add ở /kaggle/input (Teacher yolo11x-seg + Geometric Mask Merger)
RUN_VN_DASHCAM = True
VN_DASHCAM_HF_REPO = "OpenHay/zalo-ai-2025-public-test-data-v2"
VN_DASHCAM_LOCAL_INPUT = ""  # Điền đường dẫn /kaggle/input/... nếu bạn tự upload video/ảnh VN
DASHCAM_MAX_FRAMES = 400
TEACHER_WEIGHTS = "yolo11x-seg.pt"

# Nguồn 3: Kaggle Input Datasets (tự dò trong /kaggle/input)
RUN_KAGGLE_INPUT_PUBLIC = True
IDD_MAX_IMAGES = 1800
BDD_MAX_IMAGES = 1200   # Tự động ưu tiên lọc cảnh đêm (night) và mưa (rainy)
COCO_MAX_IMAGES = 1000

# ---- Tăng cường dữ liệu & Chia Golden Val ----
RUN_CROP_ZOOM = True
RUN_COPY_PASTE = True
COPY_PASTE_COUNT = 250
RUN_WEATHER_AUG = True
WEATHER_COUNT = 200
AUTO_GOLDEN_VAL_TARGET = 250  # Nếu chưa tạo sẵn GOLDEN_VAL_DIR, tự trích an toàn TRƯỚC bước augment với seed=39

print(f"{TEAM_NAME} | TEAM_ID={TEAM_ID} | IS_KAGGLE={IS_KAGGLE}")


In [ ]:
#@title 2. Cài đặt gói chuẩn & giải nén mã nguồn nhúng vào REPO_DIR
import base64
import importlib.metadata
import json
import subprocess
import sys
import zlib

_BUNDLE_B64 = (
'...'
)
EMBEDDED_FILES = json.loads(zlib.decompress(base64.b64decode(_BUNDLE_B64)).decode("utf-8"))

for rel_path, content in EMBEDDED_FILES.items():
    target = REPO_DIR / rel_path
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content, encoding="utf-8")

print(f"Đã giải nén {len(EMBEDDED_FILES)} tệp mã nguồn & cấu hình vào: {REPO_DIR.resolve()}")

if INSTALL_DEPS:
    specs = [
        "ultralytics==8.4.214",
        "onnx==1.23.2",
        "onnxslim==0.1.99",
        "onnxruntime==1.32.0",
        "pycocotools",
        "shapely",
        "huggingface_hub",
        "pyyaml",
    ]
    print("Đang cài đặt thư viện chuẩn cuộc thi...")
    proc = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", *specs], capture_output=True, text=True)
    if proc.returncode != 0:
        print(proc.stdout[-1000:], proc.stderr[-1000:])
        raise RuntimeError(f"pip install failed with code {proc.returncode}")

for pkg in ("ultralytics", "onnx", "onnxslim", "onnxruntime", "pycocotools", "shapely"):
    try:
        print(f"  - {pkg}: {importlib.metadata.version(pkg)}")
    except importlib.metadata.PackageNotFoundError:
        print(f"  - {pkg}: NOT INSTALLED")

import torch
DEVICE = "0" if torch.cuda.is_available() else "cpu"
print("GPU:", torch.cuda.get_device_name(0) if DEVICE == "0" else "CPU (CẢNH BÁO: Rất chậm!)")

repo_str = str(REPO_DIR.resolve())
if repo_str not in sys.path:
    sys.path.insert(0, repo_str)
os.chdir(repo_str)
TMP_RAW_DIR.mkdir(parents=True, exist_ok=True)
STAGED_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
#@title 3. Tự động dò các Public Dataset đã gắn ở /kaggle/input (IDD, BDD100K, COCO)
def discover_kaggle_inputs() -> dict[str, dict[str, str]]:
    found: dict[str, dict[str, str]] = {}
    input_root = Path("/kaggle/input")
    if not input_root.exists():
        print("Không thấy /kaggle/input (đang chạy ngoài Kaggle).")
        return found

    print("Các thư mục trong /kaggle/input:")
    for d in sorted(input_root.iterdir()):
        if d.is_dir():
            print("  -", d)

    # 1. IDD (gtFine polygons)
    for gtfine_dir in sorted(input_root.rglob("gtFine")):
        if gtfine_dir.is_dir() and any(gtfine_dir.rglob("*_polygons.json")):
            cand_imgs = gtfine_dir.parent / "leftImg8bit"
            if not cand_imgs.is_dir():
                cand_imgs = gtfine_dir.parent / "images"
            if not cand_imgs.is_dir():
                alt = [p for p in gtfine_dir.parent.rglob("leftImg8bit") if p.is_dir()]
                if alt:
                    cand_imgs = alt[0]
            if cand_imgs.is_dir():
                found["idd"] = {"images_dir": str(cand_imgs), "labels_dir": str(gtfine_dir)}
                break

    # 2. BDD100K (poly2d json hoặc box2d detection json)
    for jf in sorted(input_root.rglob("*.json")):
        name_low = jf.name.lower()
        if "bdd100k" in str(jf).lower() and ("train" in name_low or "ins_seg" in name_low or "labels" in name_low):
            for sub in ("images/100k/train", "100k/train", "images/train", "images"):
                cand = jf.parent.parent / sub
                if cand.is_dir():
                    found["bdd100k"] = {"images_dir": str(cand), "labels_json": str(jf)}
                    break
            if "bdd100k" in found:
                break

    # 3. COCO 2017
    for jf in sorted(input_root.rglob("instances_train*.json")):
        for sub in ("train2017", "images/train2017", "images/train", "images"):
            cand = jf.parent.parent / sub
            if cand.is_dir():
                found["coco"] = {"images_dir": str(cand), "ann_json": str(jf)}
                break
        if "coco" in found:
            break

    print("Kết quả tự dò public datasets:", json.dumps(found, indent=2, ensure_ascii=False))
    return found


DISCOVERED_INPUTS = discover_kaggle_inputs() if RUN_KAGGLE_INPUT_PUBLIC else {}


In [ ]:
#@title 4. Luồng 1: Tải Biển báo Việt Nam (HuggingFace) -> SAM 2 tạo mask không dính cột
import random
import shutil
from huggingface_hub import snapshot_download
from src.box_to_mask_sam2 import UltralyticsSAM2Predictor, convert_bbox_dataset
from src.seg_utils import transfer_image
from src.validate_dataset import IMG_EXTS

STAGED_SOURCES: list[Path] = []

if RUN_VNTS_HF:
    vnts_raw = TMP_RAW_DIR / "vnts_hf"
    shutil.rmtree(vnts_raw, ignore_errors=True)
    print(f"Đang tải {VNTS_HF_REPO} về {vnts_raw}...")
    snapshot_download(repo_id=VNTS_HF_REPO, repo_type="dataset", local_dir=str(vnts_raw))

    img_candidates = [p for p in sorted(vnts_raw.rglob("images")) if p.is_dir() and any(x.suffix.lower() in IMG_EXTS for x in p.rglob("*") if x.is_file())]
    if not img_candidates:
        src_img_root = vnts_raw
    else:
        train_cands = [p for p in img_candidates if "train" in str(p).lower()]
        src_img_root = train_cands[0] if train_cands else img_candidates[0]

    all_vnts_imgs = sorted(p for p in src_img_root.rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXTS)
    rng = random.Random(TEAM_ID)
    if VNTS_MAX_IMAGES and len(all_vnts_imgs) > VNTS_MAX_IMAGES:
        selected_imgs = sorted(rng.sample(all_vnts_imgs, VNTS_MAX_IMAGES))
    else:
        selected_imgs = all_vnts_imgs

    vnts_sub = TMP_RAW_DIR / "vnts_subset"
    shutil.rmtree(vnts_sub, ignore_errors=True)
    (vnts_sub / "images").mkdir(parents=True)
    (vnts_sub / "labels").mkdir(parents=True)
    for idx, im in enumerate(selected_imgs):
        unique_stem = f"{idx:05d}_{im.stem}"
        transfer_image(im, vnts_sub / "images" / f"{unique_stem}{im.suffix.lower()}", mode="hardlink")
        lbl_cand = Path(str(im).replace(f"{os.sep}images{os.sep}", f"{os.sep}labels{os.sep}")).with_suffix(".txt")
        if not lbl_cand.is_file():
            lbl_cand = im.with_suffix(".txt")
        if not lbl_cand.is_file():
            matches = list(vnts_raw.rglob(f"{im.stem}.txt"))
            lbl_cand = matches[0] if matches else None
        if lbl_cand and lbl_cand.is_file():
            shutil.copy2(lbl_cand, vnts_sub / "labels" / f"{unique_stem}.txt")
        else:
            (vnts_sub / "labels" / f"{unique_stem}.txt").write_text("", encoding="utf-8")

    vnts_out = STAGED_DIR / "vnts_sam2"
    shutil.rmtree(vnts_out, ignore_errors=True)
    print(f"Đang chạy SAM2 ({SAM2_MODEL}) trên {len(selected_imgs)} ảnh biển báo VN...")
    sam_pred = UltralyticsSAM2Predictor(model_name=SAM2_MODEL, device=DEVICE)
    vnts_summary = convert_bbox_dataset(
        images_dir=vnts_sub / "images",
        labels_dir=vnts_sub / "labels",
        out_dir=vnts_out,
        predictor=sam_pred,
        split="train",
        default_class=9,
        registry_path=REPO_DIR / "datasets_registry.md",
        source_name="star092304_vn_signs",
        dataset_url=f"https://huggingface.co/datasets/{VNTS_HF_REPO}",
        license_name="Open-source",
    )
    print("Kết quả VNTS -> SAM2:", json.dumps(vnts_summary, indent=2, ensure_ascii=False))
    STAGED_SOURCES.append(vnts_out)
    shutil.rmtree(vnts_raw, ignore_errors=True)
    shutil.rmtree(vnts_sub, ignore_errors=True)
else:
    print("Bỏ qua RUN_VNTS_HF.")


In [ ]:
#@title 5. Luồng 2: Chuyển IDD / BDD100K (đêm & mưa) / COCO bằng Geometric Mask Merger
from src.box_to_mask_sam2 import UltralyticsSAM2Predictor, convert_bbox_dataset
from src.map_and_merge_public import apply_geometric_mask_merger, convert_public_source, load_class_mapping
from src.seg_utils import SegInstance, read_yolo_seg_file, transfer_image, write_yolo_seg_label
from src.validate_dataset import image_size

if "idd" in DISCOVERED_INPUTS:
    idd_info = DISCOVERED_INPUTS["idd"]
    idd_out = STAGED_DIR / "idd_train"
    shutil.rmtree(idd_out, ignore_errors=True)
    print("Đang chuyển đổi IDD...")
    idd_res = convert_public_source(
        source_name="idd",
        fmt="idd",
        images_dir=Path(idd_info["images_dir"]),
        labels_path=Path(idd_info["labels_dir"]),
        out_dir=idd_out,
        split="train",
        exclude_dir=GOLDEN_VAL_DIR if GOLDEN_VAL_DIR.exists() else None,
        max_images=IDD_MAX_IMAGES,
        registry_path=REPO_DIR / "datasets_registry.md",
        dataset_url="http://idd.insaan.iiit.ac.in/",
        license_name="Academic / Research",
    )
    print("IDD:", json.dumps(idd_res, indent=2, ensure_ascii=False))
    if idd_res["kept_images"] > 0:
        STAGED_SOURCES.append(idd_out)

if "bdd100k" in DISCOVERED_INPUTS:
    bdd_info = DISCOVERED_INPUTS["bdd100k"]
    bdd_lbl_json = Path(bdd_info["labels_json"])
    bdd_imgs_dir = Path(bdd_info["images_dir"])
    bdd_out = STAGED_DIR / "bdd_night_rain"
    shutil.rmtree(bdd_out, ignore_errors=True)

    sample_items = json.loads(bdd_lbl_json.read_text(encoding="utf-8"))
    frames = sample_items if isinstance(sample_items, list) else sample_items.get("frames", [])
    has_poly2d = any("poly2d" in lbl for it in frames[:50] for lbl in it.get("labels", []))

    if has_poly2d:
        print("BDD100K có poly2d -> chạy map_and_merge_public (lọc night & rainy)...")
        for key, kw in (("night", {"time_filter": {"night"}}), ("rainy", {"weather_filter": {"rainy"}})):
            sub_out = STAGED_DIR / f"bdd_{key}"
            res = convert_public_source(
                source_name="bdd100k",
                fmt="bdd100k",
                images_dir=bdd_imgs_dir,
                labels_path=bdd_lbl_json,
                out_dir=sub_out,
                split="train",
                exclude_dir=GOLDEN_VAL_DIR if GOLDEN_VAL_DIR.exists() else None,
                max_images=BDD_MAX_IMAGES // 2,
                registry_path=REPO_DIR / "datasets_registry.md",
                dataset_url="https://github.com/bdd100k/bdd100k",
                license_name="BDD100K License",
                **kw,
            )
            if res["kept_images"] > 0:
                STAGED_SOURCES.append(sub_out)
    else:
        print("BDD100K là bản box2d -> lọc ảnh night/rainy rồi nâng cấp lên YOLO-seg qua SAM2...")
        map_cfg = load_class_mapping(REPO_DIR / "configs/class_mapping.yaml")
        bdd_map = {str(k).lower(): int(v) for k, v in map_cfg["sources"]["bdd100k"]["direct_map"].items()}
        bdd_tmp = TMP_RAW_DIR / "bdd_box_subset"
        shutil.rmtree(bdd_tmp, ignore_errors=True)
        (bdd_tmp / "images").mkdir(parents=True)
        (bdd_tmp / "labels").mkdir(parents=True)
        kept = 0
        for item in frames:
            if kept >= BDD_MAX_IMAGES:
                break
            attrs = item.get("attributes", {}) or {}
            is_hard = str(attrs.get("timeofday", "")).lower() == "night" or str(attrs.get("weather", "")).lower() in ("rainy", "foggy")
            if not is_hard:
                continue
            im_p = bdd_imgs_dir / Path(str(item.get("name", ""))).name
            if not im_p.is_file():
                continue
            w, h = image_size(im_p)
            lines = []
            for lbl in item.get("labels", []):
                cat = str(lbl.get("category", "")).strip().lower()
                b2d = lbl.get("box2d")
                if cat not in bdd_map or not b2d:
                    continue
                xc = 0.5 * (b2d["x1"] + b2d["x2"]) / w
                yc = 0.5 * (b2d["y1"] + b2d["y2"]) / h
                bw = (b2d["x2"] - b2d["x1"]) / w
                bh = (b2d["y2"] - b2d["y1"]) / h
                if bw > 0 and bh > 0:
                    lines.append(f"{bdd_map[cat]} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")
            if lines:
                transfer_image(im_p, bdd_tmp / "images" / im_p.name, mode="hardlink")
                (bdd_tmp / "labels" / f"{im_p.stem}.txt").write_text("\n".join(lines) + "\n", encoding="utf-8")
                kept += 1
        if kept > 0:
            sam_pred = UltralyticsSAM2Predictor(model_name=SAM2_MODEL, device=DEVICE)
            convert_bbox_dataset(
                bdd_tmp / "images", bdd_tmp / "labels", bdd_out, predictor=sam_pred, split="train",
                registry_path=REPO_DIR / "datasets_registry.md", source_name="bdd100k_night_rain_sam2",
                dataset_url="https://github.com/bdd100k/bdd100k", license_name="BDD100K License",
            )
            lbl_dir = bdd_out / "labels" / "train"
            for lf in sorted(lbl_dir.glob("*.txt")):
                raw_insts = read_yolo_seg_file(lf)
                tagged = [SegInstance(cls=it.cls, pts=it.pts, source_label="rider" if it.cls == 8 else "") for it in raw_insts]
                merged, _, _ = apply_geometric_mask_merger(tagged, map_cfg["geometric_merge_rules"], {"rider"}, lf.stem)
                write_yolo_seg_label(lf, [(it.cls, it.pts) for it in merged])
            STAGED_SOURCES.append(bdd_out)
        shutil.rmtree(bdd_tmp, ignore_errors=True)

if "coco" in DISCOVERED_INPUTS:
    coco_info = DISCOVERED_INPUTS["coco"]
    coco_out = STAGED_DIR / "coco_traffic"
    shutil.rmtree(coco_out, ignore_errors=True)
    coco_res = convert_public_source(
        source_name="coco", fmt="coco",
        images_dir=Path(coco_info["images_dir"]), labels_path=Path(coco_info["ann_json"]),
        out_dir=coco_out, split="train",
        exclude_dir=GOLDEN_VAL_DIR if GOLDEN_VAL_DIR.exists() else None,
        max_images=COCO_MAX_IMAGES,
        registry_path=REPO_DIR / "datasets_registry.md",
        dataset_url="https://cocodataset.org", license_name="CC BY 4.0",
    )
    print("COCO:", json.dumps(coco_res, indent=2, ensure_ascii=False))
    if coco_res["kept_images"] > 0:
        STAGED_SOURCES.append(coco_out)


In [ ]:
#@title 6. Luồng 3: Dashcam Việt Nam + Teacher Pseudo-Labeling (yolo11x-seg + Geometric Merger)
import zipfile
from src.prepare_vn_frames import UltralyticsTeacherPredictor, extract_frames_from_sources, pseudo_label_folder

if RUN_VN_DASHCAM:
    vn_src = Path(VN_DASHCAM_LOCAL_INPUT) if VN_DASHCAM_LOCAL_INPUT else (TMP_RAW_DIR / "vn_dashcam")
    if not VN_DASHCAM_LOCAL_INPUT:
        shutil.rmtree(vn_src, ignore_errors=True)
        print(f"Đang tải dashcam VN {VN_DASHCAM_HF_REPO}...")
        snapshot_download(repo_id=VN_DASHCAM_HF_REPO, repo_type="dataset", local_dir=str(vn_src))
        for zp in sorted(vn_src.rglob("*.zip")):
            with zipfile.ZipFile(zp) as zf:
                zf.extractall(vn_src / zp.stem)

    vn_staged = STAGED_DIR / "vn_dashcam_pseudo"
    shutil.rmtree(vn_staged, ignore_errors=True)
    ext_res = extract_frames_from_sources(
        input_path=vn_src,
        out_img_dir=vn_staged / "images" / "train",
        exclude_dir=GOLDEN_VAL_DIR if GOLDEN_VAL_DIR.exists() else None,
        stride_sec=1.5,
        min_sharpness=45.0,
        max_frames=DASHCAM_MAX_FRAMES,
    )
    print("Trích frame VN:", json.dumps({k: v for k, v in ext_res.items() if k != "image_paths"}, indent=2))
    if ext_res["frames_kept"] > 0:
        teacher = UltralyticsTeacherPredictor(weights=TEACHER_WEIGHTS, conf=0.25, imgsz=1280, device=DEVICE)
        pl_res = pseudo_label_folder(
            images_dir=vn_staged / "images" / "train",
            labels_out_dir=vn_staged / "labels" / "train",
            predictor=teacher,
            source_mapping_name="coco",
            registry_path=REPO_DIR / "datasets_registry.md",
            dataset_name="vn_dashcam_openhay",
            dataset_url=f"https://huggingface.co/datasets/{VN_DASHCAM_HF_REPO}",
            license_name="Open-source / Research",
        )
        print("Pseudo-label VN:", json.dumps(pl_res, indent=2, ensure_ascii=False))
        STAGED_SOURCES.append(vn_staged)
    else:
        print("Lưu ý: Không trích được ảnh/video trực tiếp từ nguồn dashcam; nếu repo HF dùng định dạng khác hãy trỏ VN_DASHCAM_LOCAL_INPUT tới thư mục ảnh/video VN.")
    if not VN_DASHCAM_LOCAL_INPUT:
        shutil.rmtree(vn_src, ignore_errors=True)


In [ ]:
#@title 7. Khóa Golden Val Set TRƯỚC khi Augment (Đảm bảo 0% rò rỉ dữ liệu sang Train)
from src.seg_utils import read_yolo_seg_file, transfer_image, write_yolo_seg_label
from src.validate_dataset import IMG_EXTS

if not STAGED_SOURCES:
    raise RuntimeError("Chưa có nguồn dữ liệu nào được tạo ở các ô trên. Hãy bật ít nhất 1 nguồn dữ liệu!")

if not GOLDEN_VAL_DIR.exists() or not any((GOLDEN_VAL_DIR / "images").rglob("*")):
    print(f"Tạo Golden Val cố định (~{AUTO_GOLDEN_VAL_TARGET} ảnh, seed={TEAM_ID}) trước khi chạy Augment...")
    val_img_dir = GOLDEN_VAL_DIR / "images" / "val"
    val_lbl_dir = GOLDEN_VAL_DIR / "labels" / "val"
    val_img_dir.mkdir(parents=True, exist_ok=True)
    val_lbl_dir.mkdir(parents=True, exist_ok=True)

    real_sources = [s for s in STAGED_SOURCES if not s.name.startswith("aug_")]
    per_src_quota = max(20, AUTO_GOLDEN_VAL_TARGET // max(1, len(real_sources)))
    rng = random.Random(TEAM_ID)
    total_val = 0
    for src_dir in real_sources:
        imgs = sorted(p for p in (src_dir / "images" / "train").glob("*") if p.suffix.lower() in IMG_EXTS)
        chosen = rng.sample(imgs, min(per_src_quota, len(imgs)))
        for im_p in chosen:
            lbl_p = src_dir / "labels" / "train" / f"{im_p.stem}.txt"
            insts = read_yolo_seg_file(lbl_p)
            transfer_image(im_p, val_img_dir / f"{src_dir.name}_{im_p.name}", mode="copy")
            write_yolo_seg_label(val_lbl_dir / f"{src_dir.name}_{im_p.stem}.txt", [(it.cls, it.pts) for it in insts])
            im_p.unlink(missing_ok=True)
            lbl_p.unlink(missing_ok=True)
            total_val += 1
    print(f"Đã khóa cứng {total_val} ảnh vào {GOLDEN_VAL_DIR} (khuyến nghị duyệt tay 100% trước Iteration 2).")
else:
    print(f"Sử dụng Golden Val có sẵn tại: {GOLDEN_VAL_DIR}")


In [ ]:
#@title 8. Tăng cường dữ liệu Data-Centric (chỉ trên Train): Crop-Zoom + Copy-Paste + Weather
from src.augment_and_crop import generate_copypaste_dataset, generate_crop_zoom_dataset, generate_weather_dataset

base_sources = [s for s in STAGED_SOURCES if not s.name.startswith("aug_")]

if RUN_CROP_ZOOM and (STAGED_DIR / "vnts_sam2").exists():
    cz_out = STAGED_DIR / "aug_cropzoom"
    shutil.rmtree(cz_out, ignore_errors=True)
    print("Crop-Zoom biển báo nhỏ:", generate_crop_zoom_dataset(STAGED_DIR / "vnts_sam2", cz_out, split="train", crop_size=640, seed=TEAM_ID))
    if any((cz_out / "images" / "train").glob("*")):
        STAGED_SOURCES.append(cz_out)

if RUN_COPY_PASTE and base_sources:
    bank_src = (STAGED_DIR / "idd_train") if (STAGED_DIR / "idd_train").exists() else base_sources[0]
    bg_src = (STAGED_DIR / "vn_dashcam_pseudo") if (STAGED_DIR / "vn_dashcam_pseudo").exists() else base_sources[-1]
    cp_out = STAGED_DIR / "aug_copypaste"
    shutil.rmtree(cp_out, ignore_errors=True)
    cp_res = generate_copypaste_dataset(bank_src, bg_src, cp_out, split="train", rare_classes=(3, 6, 7, 9, 10), n_output=COPY_PASTE_COUNT, seed=TEAM_ID)
    print("Copy-Paste lớp hiếm:", cp_res)
    if cp_res["images_written"] > 0:
        STAGED_SOURCES.append(cp_out)

if RUN_WEATHER_AUG and (STAGED_DIR / "vn_dashcam_pseudo").exists():
    we_out = STAGED_DIR / "aug_weather"
    shutil.rmtree(we_out, ignore_errors=True)
    we_res = generate_weather_dataset(STAGED_DIR / "vn_dashcam_pseudo", we_out, split="train", max_images=WEATHER_COUNT, seed=TEAM_ID)
    print("Weather Augmentation (đêm/mưa/nắng gắt):", we_res)
    if we_res["images_written"] > 0:
        STAGED_SOURCES.append(we_out)


In [ ]:
#@title 9. Gộp Master Dataset, QC Tầng 1 & Xuất du_lieu_v1.zip
from src.build_master import build_master_dataset
from src.validate_dataset import load_competition_config, print_qc_summary, run_qc

shutil.rmtree(MASTER_DIR, ignore_errors=True)
master_summary = build_master_dataset(
    train_sources=STAGED_SOURCES,
    golden_val_dir=GOLDEN_VAL_DIR,
    master_dir=MASTER_DIR,
    zip_out=OUT_ZIP_PATH,
)
print("Kết quả Build Master:", json.dumps(master_summary, indent=2, ensure_ascii=False))

cfg = load_competition_config(REPO_DIR / "configs/competition.yaml")
qc_rep = run_qc(MASTER_DIR, cfg, out_dir=Path("/kaggle/working/qc_master") if IS_KAGGLE else Path("work_kaggle/qc_master"))
print_qc_summary(qc_rep, cfg)

reg_out = Path("/kaggle/working/datasets_registry.md") if IS_KAGGLE else (MASTER_DIR.parent / "datasets_registry.md")
shutil.copy2(REPO_DIR / "datasets_registry.md", reg_out)
print(f"\n[XONG] Đã tạo {OUT_ZIP_PATH} ({OUT_ZIP_PATH.stat().st_size / 2**20:.1f} MiB) và thư mục {MASTER_DIR}.")
print("Bước tiếp theo: Import train_and_export.ipynb (TEAM_ID = 39), đặt DATASET_PATH =", OUT_ZIP_PATH)


### Bước 10 (Tùy chọn — Vòng 2 trở đi): Chạy Tier-2 Confident Learning QC tìm nhãn bẩn
Sau khi đã chạy `train_and_export.ipynb` xong lượt đầu và có `/kaggle/working/outputs/submission.zip`, chạy ô bên dưới để tự động tìm **nhãn bị sót**, **nhãn gán nhầm lớp (`Car/Bus`, `Truck/Other vehicle`, `Motorcycle/Motorcyclist`)**, và **mask lệch biên**:


In [ ]:
#@title 10. (Chạy sau vòng 1) Confident Learning QC trên tập Train & Chấm Golden Val
RUN_TIER2_QC = False
SUBMISSION_ZIP_PATH = "/kaggle/working/outputs/submission.zip"

if RUN_TIER2_QC:
    from src.eval_golden import run_eval_or_qc
    from src.selfcheck_bridge import materialize_selfcheck

    sub_p = Path(SUBMISSION_ZIP_PATH)
    eval_work = Path("/kaggle/working/_selfcheck_eval") if IS_KAGGLE else Path("work_kaggle/_selfcheck_eval")
    sc = materialize_selfcheck(REPO_DIR / "train_and_export.ipynb", eval_work)
    summary = sc.package.inspect_zip(str(sub_p), sc.config)
    onnx_p = sc.package.extract_model(str(sub_p), eval_work / "extracted", sc.config, summary)

    val_rep = run_eval_or_qc(
        model_path=onnx_p,
        images_dir=MASTER_DIR / "images" / "val",
        labels_dir=MASTER_DIR / "labels" / "val",
        out_dir=eval_work / "eval_golden",
        mode="eval",
    )
    print("Golden Val Score:", val_rep["score_summary"]["score"])
    qc_rep = run_eval_or_qc(
        model_path=onnx_p,
        images_dir=MASTER_DIR / "images" / "train",
        labels_dir=MASTER_DIR / "labels" / "train",
        out_dir=eval_work / "qc_confident_train",
        mode="qc-train",
    )
    print("Confident QC suspects:", {k: len(v) for k, v in qc_rep["confident_qc"].items()})
